In [ ]:
import os
from tqdm import tqdm
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1202 21:13:40.281000 38236 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
# === Configuration ===
DOCUMENTS_PATH = "data"
CHROMA_HOST = "localhost"
CHROMA_PORT = 8000
CHROMA_COLLECTION = "tax_documentation"
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
BATCH_SIZE = 16
NUMBER_OF_DOCUMENT = 100

In [3]:
# === Load text documents ===
def load_text_documents(path):
	documents = []
	count = 0
	for filename in os.listdir(path):
		if count >= NUMBER_OF_DOCUMENT:
			break

		if filename.endswith(".txt"):
			count += 1
			file_path = os.path.join(path, filename)
			loader = TextLoader(file_path, encoding='utf-8')
			raw_docs = loader.load()
			for doc in raw_docs:
				# Normalize content
				content = doc.page_content.replace("\n", " ").lower().strip()
				doc.page_content = content
				documents.append(doc)
	return documents

print("Loading text documents...")
raw_documents = load_text_documents(DOCUMENTS_PATH)
print(f"Total documents found: {len(raw_documents)}")

Loading text documents...
Total documents found: 100


In [4]:
subset_idx = len(raw_documents)
raw_documents = raw_documents[:subset_idx]
print(f"Documents selected for embedding: {len(raw_documents)}")

Documents selected for embedding: 100


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=50,
    separators=[
        "\nI.",
        "\nII.",
        "\nIII.",
        "\nV.",
        "\nVI.",
        "\nVII.",
        "\n1",
        "\n2",
        "\n3",
        "\n4",
        "\n5",
        "\n6",
        "\n7",
        "\n8",
        "\n9",
        "\na.",
        "\nb.",
        "\nc.",
        "\nd.",
        "\ne.",
        "\nf.",
        "\ng.",
        "\nh.",
        "\n\n",           # tách đoạn theo 2 newline
        ". ",             # tách theo câu
        " ",              # tách theo khoảng trắng
        ""                # fallback
    ]
)

In [7]:
texts = []
for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
    chunks = text_splitter.split_documents([doc])
    texts.extend(chunks)

print(f"Total chunks created: {len(texts)}")

Splitting documents into chunks: 100%|██████████| 100/100 [00:00<00:00, 1776.42it/s]

Total chunks created: 970


In [8]:
# === Initialize embeddings ===
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"}
)


Loading embedding model: keepitreal/vietnamese-sbert


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_38236\3384064487.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [9]:
# Connect to Chroma Docker server
client = chromadb.HttpClient(
    host=CHROMA_HOST,
    port=CHROMA_PORT
)

vectorstore = Chroma(
    client=client,
    collection_name=CHROMA_COLLECTION,
    embedding_function=embeddings
)

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to Chroma"):
    batch = texts[i:i + BATCH_SIZE]

    # Chroma will handle embeddings internally OR use yours
    batch_embeddings = embeddings.embed_documents([t.page_content for t in batch])

    vectorstore.add_texts(
        texts=[t.page_content for t in batch],
        embeddings=batch_embeddings
    )

print(f"✅ Chroma vectorstore successfully stored in collection: '{CHROMA_COLLECTION}'")
print("✅ Data is persisted inside Docker volume automatically")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_38236\1971947378.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
Embedding and adding to Chroma: 100%|██████████| 61/61 [00:19<00:00,  3.16it/s]

✅ Chroma vectorstore successfully stored in collection: 'tax_documentation'
✅ Data is persisted inside Docker volume automatically
